In [1]:
import pandas as pd
from transformers import AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def transform_papers(df):
    records = []
    for _, paper in df.iterrows():
        for section, section_paragraphs in zip(
            paper["full_text"]["section_name"],
            paper["full_text"]["paragraphs"]
        ):
            for paragraph_id, text in enumerate(section_paragraphs):
                text = text.strip()
                if text:
                    records.append({
                        "paper_id": paper["id"],
                        "title": paper["title"],
                        "section": section,
                        "paragraph_id": paragraph_id,
                        "text": text
                    })
    return pd.DataFrame(records, columns=[
        "paper_id", "title", "section", "paragraph_id", "text"
    ])


def tokenize_paragraphs(df):
    result = df.copy()
    encoded = tokenizer(
        result["text"].tolist(),
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False
    )
    result["token_ids"] = encoded["input_ids"]
    result["token_count"] = result["token_ids"].map(len)
    return result


def chunk_paragraphs(df, max_tokens=250, overlap=40, max_seq_length=None):
    if not 0 <= overlap < max_tokens:
        raise ValueError("Require 0 <= overlap < max_tokens")
    if max_seq_length is not None and max_tokens > max_seq_length:
        raise ValueError(f"Chunk size {max_tokens} exceeds model limit {max_seq_length}")

    records = []
    for _, row in df.iterrows():
        token_ids = row["token_ids"]
        for chunk_id, start in enumerate(range(0, len(token_ids), max_tokens - overlap)):
            end = min(start + max_tokens, len(token_ids))
            chunk_tokens = token_ids[start:end]
            record = row.to_dict()
            record.update({
                "text": tokenizer.decode(
                    chunk_tokens,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False
                ),
                "token_ids": chunk_tokens,
                "token_count": len(chunk_tokens),
                "chunk_id": chunk_id,
                "token_start": start,
                "token_end": end
            })
            records.append(record)
            if end == len(token_ids):
                break
    return pd.DataFrame(records)


def preprocess(df, max_tokens=250, overlap=40, max_seq_length=None):
    transformed = transform_papers(df)
    tokenized = tokenize_paragraphs(transformed)
    return chunk_paragraphs(tokenized, max_tokens=max_tokens, overlap=overlap, max_seq_length=max_seq_length)

In [6]:
train_df = pd.read_parquet("~/Downloads/qasper-rag/data/train_qasper.parquet") 
test_df = pd.read_parquet("~/Downloads/qasper-rag/data/test_qasper.parquet") 
validation_df = pd.read_parquet("~/Downloads/qasper-rag/data/validation_qasper.parquet")


In [2]:
import re

def clean_text(text):
    text = re.sub(r'\b(BIBREF|TABREF|FIGREF|SECREF|bibref|tabref|figref|secref)\d*\b', '', text)
    text = re.sub(r'\$[^$]+\$', '', text)   # remove inline LaTeX
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def transform_papers(df):
    records = []
    for _, paper in df.iterrows():
        for section, section_paragraphs in zip(
            paper["full_text"]["section_name"],
            paper["full_text"]["paragraphs"]
        ):
            for paragraph_id, text in enumerate(section_paragraphs):
                text = clean_text(text.strip())
                if text:
                    records.append({
                        "paper_id": paper["id"],
                        "title": paper["title"],
                        "section": section,
                        "paragraph_id": paragraph_id,
                        "text": text
                    })
    return pd.DataFrame(records, columns=[
        "paper_id", "title", "section", "paragraph_id", "text"
    ])

In [3]:
def preprocess(df, max_tokens=250, overlap=40, max_seq_length=None):
    transformed = transform_papers(df)
    tokenized = tokenize_paragraphs(transformed)
    chunks = chunk_paragraphs(tokenized, max_tokens=max_tokens, overlap=overlap, max_seq_length=max_seq_length)
    # Prepend section as context prefix
    chunks["text"] = chunks["section"] + ": " + chunks["text"]
    return chunks

In [4]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

model = SentenceTransformer(MODEL_NAME)
print("Token limit:", model.max_seq_length)  # 512

def embed_chunks(chunks):
    return model.encode(
        chunks["text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

def embed_queries(questions):
    return model.encode(
        [QUERY_PREFIX + q for q in questions],
        batch_size=32,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Token limit: 512


In [7]:
#re running chunking w/ larger limits 

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_chunks = preprocess(train_df, max_tokens=400, overlap=60, max_seq_length=model.max_seq_length)
validation_chunks = preprocess(validation_df, max_tokens=400, overlap=60, max_seq_length=model.max_seq_length)



[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (888 > 512). Running this sequence through the model will result in indexing errors


In [8]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(question, candidate_chunks, top_k=5):
    pairs = [(question, row["text"]) for _, row in candidate_chunks.iterrows()]
    scores = reranker.predict(pairs)
    candidate_chunks = candidate_chunks.copy()
    candidate_chunks["rerank_score"] = scores
    return candidate_chunks.sort_values("rerank_score", ascending=False).head(top_k)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\beepo\Downloadsminiconda3\envs\tuffenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\beepo\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\beepo\Downloadsminiconda3\envs\tuffenv\Lib\site-packages\huggingface_hub\fi

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [9]:
# Retrieve 20 candidates, rerank down to 5
candidates = hybrid_search(query, chunks, bm25_index, faiss_index, query_embedding, top_k=20)
final = rerank(query, candidates, top_k=5)

NameError: name 'hybrid_search' is not defined